# 🏆 Competición Box-Jenkins — `ts_w5`
## Notebook del Estudiante

---

| | |
|---|---|
| **Nombre / Grupo** | _(escribe aquí)_ |
| **Fecha de entrega** | _(escribe aquí)_ |

---

### Instrucciones

Aplica la **metodología Box-Jenkins completa** a la serie temporal `ts_w5`:

1. **Identificación** — inspección visual, contrastes de estacionariedad (ADF + KPSS),
   diferenciación si es necesario, análisis ACF/PACF → orden tentativo (p, d, q)
2. **Estimación** — ajustar ARIMA(p, d, q), comparar candidatos por AIC/BIC
3. **Diagnóstico** — residuos, contraste de Ljung-Box

### Puntuación

| Criterio | Puntos |
|----------|--------|
| Orden (p, d, q) correcto | 5 |
| Residuos pasan Ljung-Box (p > 0.05 en todos los retardos) | 5 |
| **Total** | **10** |

> ⚠️ La serie puede ser **no estacionaria** (d > 0). Debes detectarlo con
> los contrastes y diferenciar antes de identificar p y q.


## ⚙️ Configuración

In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
import warnings
warnings.filterwarnings("ignore")
from importlib.resources import open_binary

from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.gofplots import qqplot
from scipy import stats

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 11})

ROSA = "#DB2777"; ROJO = "#DC2626"; VERDE = "#16A34A"; AZUL = "#2563EB"

def load_data_ts_w5():
    with open_binary("data", "ts_w5.npy") as f:
        return np.load(f)

# Carga como Serie con índice numérico (sin fechas)
ts_w5 = pd.Series(load_data_ts_w5(), name="ts_w5")

print(f"Serie cargada: {len(ts_w5)} observaciones")
print(ts_w5.describe().to_string())


---
## Etapa 1 · Identificación

### 1.1 Inspección visual


In [20]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ts_w5.values, color=ROSA, lw=1.4, marker="o", ms=3, label="ts_w5")
ax.axhline(ts_w5.mean(), color=ROJO, ls="--", lw=1.2,
           label=f"Media = {ts_w5.mean():.3f}")
ax.fill_between(range(len(ts_w5)),
                ts_w5.mean() - 2*ts_w5.std(),
                ts_w5.mean() + 2*ts_w5.std(),
                alpha=0.08, color=ROJO, label="±2σ")
ax.set_title("ts_w5 — Serie original", fontweight="bold")
ax.set_xlabel("Observación"); ax.set_ylabel("Valor")
ax.legend()
plt.tight_layout()
plt.show()


Serie cargada: 54 observaciones
count      54.000000
mean      413.355556
std       605.309099
min       100.400000
25%       147.725000
50%       187.550000
75%       243.500000
max      1996.000000


**📝 Observaciones visuales:**

_(¿Hay tendencia? ¿La varianza parece constante? ¿Hay ciclos visibles?)_

```
...
```


### 1.2 Contrastes de estacionariedad

Aplicamos dos contrastes complementarios:

| Contraste | $H_0$ | Rechazar $H_0$ implica… |
|-----------|--------|------------------------|
| **ADF** | Existe raíz unitaria (no estacionaria) | Serie **es** estacionaria |
| **KPSS** | Serie es estacionaria | Serie **no es** estacionaria |


In [ ]:
# ── ADF ──────────────────────────────────────────────────────────────────────
adf = adfuller(ts_w5, autolag="AIC")
print(f"[ADF]  Estadístico = {adf[0]:.4f}   p-valor = {adf[1]:.4f}")
for k, v in adf[4].items():
    print(f"       Valor crítico {k}: {v:.4f}")
adf_concl = "ESTACIONARIA ✓" if adf[1] < 0.05 else "NO ESTACIONARIA ✗"
print(f"       → {adf_concl}")

# ── KPSS ─────────────────────────────────────────────────────────────────────
kpss_res = kpss(ts_w5, regression="c", nlags="auto")
print(f"\n[KPSS] Estadístico = {kpss_res[0]:.4f}   p-valor = {kpss_res[1]:.4f}")
for k, v in kpss_res[3].items():
    print(f"       Valor crítico {k}: {v:.4f}")
kpss_concl = "ESTACIONARIA ✓" if kpss_res[1] > 0.05 else "NO ESTACIONARIA ✗"
print(f"       → {kpss_concl}")


**📝 Interpretación conjunta:**

| Contraste | Estadístico | p-valor | Conclusión |
|-----------|-------------|---------|------------|
| ADF  | _(rellena)_ | _(rellena)_ | _(rellena)_ |
| KPSS | _(rellena)_ | _(rellena)_ | _(rellena)_ |

**¿Es necesario diferenciar?** _(Sí / No — justifica)_

```
...
```


### 1.3 Diferenciación (si es necesaria)

In [ ]:
# TODO: cambia d según tus contrastes  (0 = estacionaria,  1 = una diferencia, ...)
d = 0   # <── CAMBIA ESTE VALOR

if d > 0:
    ts_w5_est = ts_w5.diff(d).dropna().reset_index(drop=True)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(ts_w5.values,     color=ROSA,  lw=1.3, marker="o", ms=2.5)
    axes[0].axhline(ts_w5.mean(),  color=ROJO, ls="--", lw=1)
    axes[0].set_title("Original"); axes[0].set_ylabel("Valor")

    axes[1].plot(ts_w5_est.values, color=VERDE, lw=1.3, marker="o", ms=2.5)
    axes[1].axhline(ts_w5_est.mean(), color=ROJO, ls="--", lw=1)
    axes[1].set_title(f"Diferenciada (d={d})"); axes[1].set_ylabel(f"∇^{d}y_t")

    plt.suptitle("ts_w5 — Diferenciación", fontweight="bold")
    plt.tight_layout(); plt.show()

    # Re-contrastar sobre la serie diferenciada
    adf_d = adfuller(ts_w5_est, autolag="AIC")
    kpss_d = kpss(ts_w5_est, regression="c", nlags="auto")
    print(f"[ADF  tras d={d}]  p-valor = {adf_d[1]:.4f}  → {'ESTACIONARIA ✓' if adf_d[1] < 0.05 else 'NO ESTACIONARIA ✗'}")
    print(f"[KPSS tras d={d}]  p-valor = {kpss_d[1]:.4f}  → {'ESTACIONARIA ✓' if kpss_d[1] > 0.05 else 'NO ESTACIONARIA ✗'}")
else:
    ts_w5_est = ts_w5.copy()
    print("d = 0 → no se aplica diferenciación")


### 1.4 ACF y PACF

#### Reglas de identificación

| Patrón ACF | Patrón PACF | Modelo |
|------------|-------------|--------|
| Se corta en lag $q$ | Decae gradualmente | **MA($q$)** |
| Decae gradualmente | Se corta en lag $p$ | **AR($p$)** |
| Decae gradualmente | Decae gradualmente | **ARMA($p$,$q$)** |


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

plot_acf(ts_w5_est, lags=20, alpha=0.05, ax=axes[0], color=ROSA,
         title="ACF muestral — ts_w5" + (f" (d={d})" if d > 0 else ""))
plot_pacf(ts_w5_est, lags=20, alpha=0.05, ax=axes[1], color=ROSA,
          title="PACF muestral — ts_w5" + (f" (d={d})" if d > 0 else ""),
          method="ywm")

for ax in axes:
    ax.set_xlabel("Retardo")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

plt.tight_layout()
plt.show()


In [ ]:
# ── Retardos significativos ───────────────────────────────────────────────────
acf_vals,  acf_ic  = acf( ts_w5_est, nlags=20, alpha=0.05)
pacf_vals, pacf_ic = pacf(ts_w5_est, nlags=20, alpha=0.05, method="ywm")

acf_sig  = [i for i in range(1, 21) if not (acf_ic[i,0]  <= 0 <= acf_ic[i,1])]
pacf_sig = [i for i in range(1, 21) if not (pacf_ic[i,0] <= 0 <= pacf_ic[i,1])]

print(f"Retardos ACF  significativos (→ MA): {acf_sig}")
print(f"Retardos PACF significativos (→ AR): {pacf_sig}")


**📝 Interpretación ACF/PACF:**

| Gráfico | Patrón observado | Interpretación |
|---------|-----------------|----------------|
| ACF  | _(rellena)_ | _(rellena)_ |
| PACF | _(rellena)_ | _(rellena)_ |

**Modelo tentativo:** ARIMA( `p = ___` , `d = ___` , `q = ___` )


---
## Etapa 2 · Estimación

### 2.1 Comparación de modelos candidatos

$$AIC = -2\ln(\hat{L}) + 2k \qquad BIC = -2\ln(\hat{L}) + k\ln(n)$$

**Valor más bajo = mejor modelo.**


In [ ]:
# TODO: ajusta la lista según lo que hayas identificado en ACF/PACF
candidatos = [(1,0), (2,0), (0,1), (0,2), (1,1), (2,1), (1,2)]

registros = []
for p, q in candidatos:
    try:
        res = ARIMA(ts_w5, order=(p, d, q), trend="n").fit()
        registros.append({"Modelo": f"ARIMA({p},{d},{q})",
                          "k": p + q + 1,
                          "LogVer": round(res.llf, 2),
                          "AIC": round(res.aic, 2),
                          "BIC": round(res.bic, 2),
                          "result": res})
    except Exception as e:
        print(f"  ARIMA({p},{d},{q}) falló: {e}")

comp = (pd.DataFrame(registros)
          .drop(columns=["result"])
          .sort_values("AIC")
          .reset_index(drop=True))
comp.index += 1
print(comp.to_string())


In [ ]:
# ── Gráfico de comparación AIC / BIC ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, crit in zip(axes, ["AIC", "BIC"]):
    df_s = comp.sort_values(crit)
    colores = [VERDE if i == 0 else "#93C5FD" for i in range(len(df_s))]
    ax.barh(df_s["Modelo"], df_s[crit], color=colores, edgecolor="white")
    ax.axvline(df_s[crit].iloc[0], color=ROJO, ls="--", lw=1.2,
               label=f"Mejor {crit} = {df_s[crit].iloc[0]:.1f}")
    ax.set_xlabel(crit)
    ax.set_title(f"Comparación — {crit}", fontweight="bold")
    ax.legend(fontsize=9); ax.invert_yaxis()
plt.tight_layout(); plt.show()


In [ ]:
# TODO: introduce tu modelo final  ──────────────────────────────────────────
p_final, d_final, q_final = 1, d, 0   # <── CAMBIA ESTOS VALORES

result = ARIMA(ts_w5, order=(p_final, d_final, q_final), trend="n").fit()
nombre_modelo = f"ARIMA({p_final},{d_final},{q_final})"

print(f"Modelo seleccionado: {nombre_modelo}")
print(result.summary())


**📝 Justificación del modelo elegido:**

_(Cita el patrón ACF/PACF observado, el valor AIC/BIC y el motivo de tu elección)_

```
...
```


---
## Etapa 3 · Diagnóstico

### 3.1 Gráficos de residuos

Un modelo bien especificado deja residuos que se comportan como **ruido blanco**:
sin estructura, media ≈ 0 y distribución aproximadamente normal.


In [ ]:
residuos = result.resid

fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

# Panel 1 — residuos en el tiempo
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(residuos.values, color=ROSA, lw=1.2)
ax1.axhline(0, color=ROJO, ls="--", lw=1)
ax1.fill_between(range(len(residuos)),
                 -2*residuos.std(), 2*residuos.std(),
                 alpha=0.08, color=ROJO, label="±2σ")
ax1.set_title("(1) Residuos en el tiempo", fontweight="bold")
ax1.set_xlabel("Observación"); ax1.set_ylabel("Residuo")
ax1.legend(fontsize=9)

# Panel 2 — ACF de residuos
ax2 = fig.add_subplot(gs[0, 1])
plot_acf(residuos, lags=15, alpha=0.05, ax=ax2,
         title="(2) ACF de residuos", color=ROSA)
ax2.set_xlabel("Retardo")

# Panel 3 — Q-Q plot
ax3 = fig.add_subplot(gs[1, 0])
qqplot(residuos, line="s", ax=ax3,
       markerfacecolor=ROSA, markeredgecolor=ROSA, alpha=0.7)
ax3.set_title("(3) Gráfico Q-Q  (Normalidad)", fontweight="bold")

# Panel 4 — histograma
ax4 = fig.add_subplot(gs[1, 1])
ax4.hist(residuos, bins=12, density=True, color=ROSA,
         edgecolor="white", alpha=0.75, label="Residuos")
xg = np.linspace(residuos.min(), residuos.max(), 300)
ax4.plot(xg, stats.norm.pdf(xg, residuos.mean(), residuos.std()),
         color=ROJO, lw=2, label="N(μ,σ²)")
ax4.set_title("(4) Histograma", fontweight="bold")
ax4.set_xlabel("Residuo"); ax4.set_ylabel("Densidad")
ax4.legend(fontsize=9)

plt.suptitle(f"Diagnóstico de residuos — {nombre_modelo}",
             fontsize=13, fontweight="bold")
plt.show()


**📝 Interpretación de los gráficos:**

| Panel | Buen ajuste | ¿Tu modelo? |
|-------|------------|-------------|
| (1) Residuos en el tiempo | Dispersión aleatoria en torno a 0, sin patrón | _(rellena)_ |
| (2) ACF de residuos | Todas las barras dentro de la banda | _(rellena)_ |
| (3) Q-Q plot | Puntos cerca de la recta a 45° | _(rellena)_ |
| (4) Histograma | Forma de campana centrada en 0 | _(rellena)_ |


### 3.2 Contraste de Ljung-Box

$$Q(h) = n(n+2)\sum_{k=1}^{h}\frac{\hat{\rho}_k^2}{n-k} \sim \chi^2(h-p-q)$$

- $H_0$: los residuos son ruido blanco hasta el retardo $h$
- Rechazar $H_0$ → el modelo no captura toda la estructura


In [ ]:
h = min(10, len(residuos) // 5)
lb = acorr_ljungbox(residuos, lags=h, return_df=True)

print(f"Contraste de Ljung-Box — {nombre_modelo}  (retardos 1 a {h})")
print("H0: los residuos son ruido blanco")
print("-" * 48)
print(lb.to_string())

aprueba = (lb["lb_pvalue"] > 0.05).all()
msg = "APRUEBA — Residuos son ruido blanco → Modelo ADECUADO" if aprueba else "FALLA  — Autocorrelacion significativa → revisar orden"
print(f"\n{'✓' if aprueba else '✗'} {msg}")


In [ ]:
# Gráfico de p-valores
fig, ax = plt.subplots(figsize=(9, 4))
ax.stem(lb.index, lb["lb_pvalue"], linefmt=ROSA, markerfmt="o", basefmt=" ")
ax.axhline(0.05, color=ROJO, ls="--", lw=1.5, label="α = 0.05")
ax.set_title(f"p-valores Ljung-Box — {nombre_modelo}", fontweight="bold")
ax.set_xlabel("Retardo h"); ax.set_ylabel("p-valor")
ax.set_ylim(0, 1.05); ax.legend()
plt.tight_layout(); plt.show()


**📝 Conclusión del diagnóstico:**

| Check | Resultado | Comentario |
|-------|-----------|------------|
| Residuos sin patrón visual | _(Sí / No)_ | |
| ACF de residuos OK | _(Sí / No)_ | |
| Ljung-Box aprueba (todos los retardos) | _(Sí / No)_ | |

**Conclusión final:** _(¿Es el modelo adecuado? Si no, ¿qué cambiarías?)_

```
...
```


---
## 📋 Resumen Final


In [ ]:
print("═" * 55)
print("  RESULTADO FINAL")
print("═" * 55)
print(f"  Serie         : ts_w5")
print(f"  Modelo final  : {nombre_modelo}")
print(f"  AIC           : {result.aic:.4f}")
print(f"  BIC           : {result.bic:.4f}")
print()
print("  Parámetros estimados:")
for nombre_p, val in result.params.items():
    ci = result.conf_int().loc[nombre_p]
    print(f"    {nombre_p:<14} = {val:>8.4f}   IC 95% [{ci[0]:.4f}, {ci[1]:.4f}]")
print()
print(f"  Ljung-Box      : {'✓ APRUEBA' if aprueba else '✗ FALLA'}")
print(f"  Orden (p,d,q)  : ({p_final}, {d_final}, {q_final})")
print("═" * 55)
